### MobileNetV3 GroupNorm investigation
Following up on requests from the review: the cause of MobileNetV3's failure to learn has been identified and fixed.

Root cause. Opacus requires replacing BatchNorm with GroupNorm for privacy compliance before it can wrap a model in DP-SGD. ModuleValidator.fix() does this automatically, but it creates the new GroupNorm layers with default weights rather than carrying over anything from the pretrained BatchNorm layers. When these layers fall inside the frozen part of the backbone, those default, untrained values are locked in for the whole run and never get a chance to adjust, so the pretrained backbone's early layers are effectively feeding the rest of the network through an untrained normalization layers.

Steps taken to isolate this:

* Checked the group counts chosen by ModuleValidator.fix(), confirmed that some groups have size 1.

* Ran MobileNetV3 on CIFAR-10 without Opacus at all (no DP-SGD), keeping the Flower federated pipeline, and it converged well there.

* Applied ModuleValidator.fix() and confirmed that it introduces the model deterioration.

* Checked whether the group counts chosen by ModuleValidator.fix() were the problem (the bad divisor concern) by forcing a larger group size everywhere. This did not meaningfully improve accuracy, ruling out group size as the main cause.

* Tested unfreezing the GroupNorm weights, while keeping the rest of the backbone frozen as before. This substantially recovered accuracy (up to 70%, vs 35-45% before), showing the frozen default weights is the actual issue.

* Repeated the same test with Opacus enabled and DP noise and clipping disabled in the pipeline. Results were nearly identical to previous step, confirming that DP-SGD code works correct.

* Repeated the MobileNetV3 test that failed previous time.

##### page break

### GroupNorm substitution check
After running `ModuleValidator.fix()` on MobileNetV3, all BatchNorm2d layers were replaced with GroupNorm. This table lists, for each replaced layer: the channel count (unchanged, equals to BatchNorm2d num_features from before the fix), the num_groups value Opacus ModuleValidator chose, and the resulting group size (channels / groups).

In [ ]:
import opacus.validators, pandas, torch, torchvision.models
model = torchvision.models.mobilenet_v3_small()
model = opacus.validators.ModuleValidator.fix(model)
group_norm = []
for name, module in model.named_modules():
    if isinstance(module, torch.nn.GroupNorm):
        group_norm.append({
            "layer": name,
            "channels": module.num_channels,
            "num_groups": module.num_groups,
            "group size": module.num_channels // module.num_groups
        })
pandas.DataFrame(group_norm)

##### page break

In [ ]:
import pandas
import matplotlib.pyplot as plt

def flwr_accuracy(folder, key, title, exclude_ids = {}):
    df = pandas.read_csv(folder + "/flwr.csv")
    df = df[~df["id"].isin(exclude_ids)]
    plt.figure(figsize=(8,4))
    for id_val, g in df.groupby("id", sort=False):
        g = g.sort_values("round")
        plt.plot(g["round"], g[key], marker="o", label=str(id_val))

    plt.xlabel("round")
    plt.ylabel(key)
    plt.title(title)
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.show()

##### page break

### MobileNetV3 in Flower federation
Test of default MobileNetV3 in Flower to ensure it gives the expected accuracy.    
Unfrozen: classifier head plus the last layer of the pretrained backbone, tune_layers: 1, batch-size: 32

In [ ]:
flwr_accuracy("mnv3woo_lr", "accuracy", "MobileNetV3 in Flower, accuracy by learning rate")

In [ ]:
flwr_accuracy("mnv3woo_lr", "loss", "")

##### page break

### MobileNetV3 with applied ModuleValidator.fix(model)
The same default MobileNetV3 with applied ModuleValidator.fix(model), tune_layers: 1, batch-size: 32

In [ ]:
flwr_accuracy("mnv3_groupnorm_lr/1", "accuracy", "")

In [ ]:
flwr_accuracy("mnv3_groupnorm_lr/1", "loss", "")

##### page break

### MobileNetV3 with applied ModuleValidator.fix(model, num_groups=4)
Applying ModuleValidator.fix() results in some groups having only one channel per group. This test eliminates this edge case.

In [ ]:
flwr_accuracy("mnv3_groupnorm_lr/2", "accuracy", "")

In [ ]:
flwr_accuracy("mnv3_groupnorm_lr/2", "loss", "")

##### page break

### MobileNetV3 with applied ModuleValidator.fix(model) and all GroupNorm unfrozen
It is possible that BatchNorm2d layers lose their pretrained weights after being replaced with GroupNorm and need to be trained from scratch.    
Unfrozen: classifier head plus the last layer of the pretrained backbone plus all GroupNorm layers.

In [ ]:
flwr_accuracy("mnv3_gn2_lr", "accuracy", "")

In [ ]:
flwr_accuracy("mnv3_gn2_lr", "loss", "")

##### page break

### MobileNetV3 with all GroupNorm unfrozen in Opacus pipeline
using settings to disable noise and clipping    
noise-multiplier: 0, max-grad-norm: 1e90

In [ ]:
flwr_accuracy("mnv3_gn3_lr", "accuracy", "")

In [ ]:
flwr_accuracy("mnv3_gn3_lr", "loss", "")

##### page break

### MobileNetV3, fixed
noise-multiplier: 1.1, max-grad-norm: 1.0, learning-rate: 0.01, batch-size: 32

In [ ]:
flwr_accuracy("mn_layers/new", "accuracy", "MobileNetV3 accuracy by trainable layers")

In [ ]:
flwr_accuracy("mn_layers/new", "loss", "")